# 10 · Descarga de entrenadores de comparación

Este notebook baja de StatsBomb los eventos de otros entrenadores de Liga MX (mismas temporadas que Jardine) para **reaplicar el framework** y comparar estilos. No cambia nada del análisis de América: solo guarda los datos en `data/raw/comparacion/`.

Las credenciales se leen del notebook 00 (no se copian a este archivo). La descarga de cada entrenador tarda unos minutos.

In [ ]:
import os, re, json, warnings
warnings.filterwarnings("ignore")
import pandas as pd

# Define SB_USERNAME y SB_PASSWORD como variables de entorno antes de abrir el notebook

from statsbombpy import sb

comps = sb.competitions()
liga_mx = comps[comps.competition_name.str.contains("Liga MX", case=False, na=False)]
frames = []
for _, r in liga_mx.iterrows():
    m = sb.matches(competition_id=r.competition_id, season_id=r.season_id)
    m["season_id"] = r.season_id
    frames.append(m)
matches_df = pd.concat(frames, ignore_index=True).drop_duplicates("match_id")
print(f"Partidos de Liga MX disponibles: {len(matches_df)}")

## 1. ¿Qué entrenadores hay y cuántos partidos tienen?

Un entrenador de comparación necesita una muestra comparable a la de Jardine (129 partidos). Se cuentan los partidos por pareja entrenador-equipo.

In [ ]:
filas = []
for _, x in matches_df.iterrows():
    for lado in ["home", "away"]:
        gestores = x[f"{lado}_managers"]
        if not isinstance(gestores, str):
            continue
        for g in [g.strip() for g in gestores.split(",") if g.strip()]:
            filas.append({"entrenador": g, "equipo": x[f"{lado}_team"], "match_id": x["match_id"]})
tabla = pd.DataFrame(filas).groupby(["entrenador", "equipo"]).match_id.nunique().reset_index(name="partidos")
tabla = tabla.sort_values("partidos", ascending=False).reset_index(drop=True)
tabla.head(25)

## 2. Selección y descarga

Por defecto se eligen los 3 entrenadores (distintos de Jardine) con más partidos. Para elegir otros, escribe la lista a mano en `COMPARAR` con el formato `("Nombre del entrenador", "Equipo")` tal como salen en la tabla de arriba.

In [ ]:
COMPARAR = []  # ejemplo: [("Nombre Entrenador", "Equipo")]; si queda vacío se usan los 3 con más partidos
 
if not COMPARAR:
    candidatos = tabla[~tabla.entrenador.str.contains("Jardine", case=False)].head(3)
    COMPARAR = list(zip(candidatos.entrenador, candidatos.equipo))
print("Se descargarán:", COMPARAR)

os.makedirs("../data/raw/comparacion", exist_ok=True)

def slug(t):
    return re.sub(r"[^a-z0-9]+", "_", t.lower()).strip("_")

for entrenador, equipo in COMPARAR:
    ids = sorted({r["match_id"] for r in filas if r["entrenador"] == entrenador and r["equipo"] == equipo})
    destino = f"../data/raw/comparacion/events_{slug(entrenador)}__{slug(equipo)}.pkl.gz"
    if os.path.exists(destino):
        print(f"{entrenador} ({equipo}): ya descargado, se omite")
        continue
    eventos = []
    for i, mid in enumerate(ids):
        ev = sb.events(match_id=mid)
        ev["match_id"] = mid
        eventos.append(ev)
        if (i + 1) % 10 == 0:
            print(f"  {entrenador}: {i + 1}/{len(ids)}")
    pd.concat(eventos, ignore_index=True).to_pickle(destino)
    sub = matches_df[matches_df.match_id.isin(ids)].copy()
    sub.to_pickle(destino.replace("events_", "partidos_"))
    print(f"{entrenador} ({equipo}): {len(ids)} partidos guardados")
print("Listo")